# 01 — TRIBE verification (decision gate 17)

Track A, Colab GPU. Verifies the pinned checkpoint against
`config/checkpoints.lock`, reproduces Meta's published example, records the
environment, and writes `outputs/verification/gate17.json`.

**`scripts/run_tribe_inference.py` refuses to touch a project stimulus until
that artifact exists for the pinned revision.** Gate 17 is enforced in code,
not on a checklist.

Nothing below computes anything itself — every cell is a bootstrap step or a
call into `scripts/`.


## Bootstrap (run once per session, then **Runtime → Restart session**)

Order matters and is not cosmetic:

1. **GPU check** — Track A needs one. Runtime → Change runtime type → GPU.
2. **Mount Drive** — Colab has no persistent disk. Sessions die at 12h, on
   disconnect, or on idle.
3. **Set `HF_HOME` before importing anything from HuggingFace.** Once a HF
   module is imported the cache location is fixed for the process, and the
   ~1 GB checkpoint lands on the ephemeral runtime disk instead of Drive.
4. **Clone the repo and `uv pip install --system`** — into Colab's own
   interpreter, never `uv sync` into a separate venv the notebook cannot see.
5. **Authenticate** from Colab Secrets (🔑), never from a literal in a cell.

Everything after that is a single call into a script in `scripts/`. No project
logic lives in this notebook: a cell dies with the session, and brief §4.3
requires every result to come from a script in the repo.


In [ ]:
# 1. GPU check
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "NO GPU")


In [ ]:
# 2. Mount Drive (persistent) -- Colab's own disk is not
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
except ModuleNotFoundError:
    DRIVE_ROOT = Path("./drive_local")   # off Colab: keep the notebook runnable

PROJECT_DRIVE = DRIVE_ROOT / "NeuroTutorSim"
CACHE_ROOT = PROJECT_DRIVE / "tribe_cache"
HF_CACHE = PROJECT_DRIVE / "hf"
for d in (PROJECT_DRIVE, CACHE_ROOT, HF_CACHE):
    d.mkdir(parents=True, exist_ok=True)
print(f"cache root : {CACHE_ROOT}")
print(f"HF cache   : {HF_CACHE}")


In [ ]:
# 3. HF_HOME -- BEFORE any huggingface import in this process.
#    Set after an import, the ~1 GB checkpoint goes to the ephemeral disk and is
#    re-downloaded every session.
import os
import sys

assert not any(m.startswith(("huggingface_hub", "transformers")) for m in sys.modules), (
    "a HuggingFace module is already imported; Runtime -> Restart session and run this cell first"
)
os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
print("HF_HOME =", os.environ["HF_HOME"])


In [ ]:
# 4. Clone the repo and install the pinned Track A stack into Colab's interpreter
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/MatteoGuardamagna4/neurotutorsim.git"
REPO_DIR = Path("/content/neurotutorsim")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
# --system installs into Colab's own interpreter. `uv sync` would build a venv
# this kernel cannot import from.
subprocess.run(
    ["uv", "pip", "install", "--system", "--extra", "tribe", "--extra", "dev", "-e", "."],
    cwd=REPO_DIR,
    check=True,
)

sys.path.insert(0, str(REPO_DIR))
print("installed; Runtime -> Restart session, then continue BELOW this cell")


In [ ]:
# 5. Authenticate. HF_TOKEN comes from Colab Secrets (the key icon), never a literal.
#    `meta-llama/Llama-3.2-3B` (TRIBE's text encoder) is gated per account:
#    a read token does not grant access until Meta approves the request.
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))
print("authenticated")


## First run only: pin the revision

`config/tribe.yaml` ships with an all-zero placeholder revision. It loads (so
Track B analysis works on a laptop) but Track A refuses to use it.

Run the two cells below **once**, then paste the SHA into `config/tribe.yaml`,
commit it together with the updated `config/checkpoints.lock`, and push. After
that, skip straight to "Run gate 17".


In [ ]:
# Print the checkpoint's current commit SHA. Nothing is written.
!cd /content/neurotutorsim && python scripts/run_tribe_verification.py \
    --config config/tribe.yaml --resolve-revision


In [ ]:
# After pasting the SHA into config/tribe.yaml: record its file checksums.
# Separate and explicit on purpose -- a lock that writes itself verifies nothing.
!cd /content/neurotutorsim && python scripts/run_tribe_verification.py \
    --config config/tribe.yaml --write-lock


## Run gate 17


In [ ]:
!cd /content/neurotutorsim && python scripts/run_tribe_verification.py \
    --config config/tribe.yaml --cache-root "$CACHE_ROOT"


## Before closing the session

`outputs/verification/gate17.json` and `docs/tribe_environment.md` are the
evidence that gate 17 was cleared on this hardware. `outputs/` is gitignored,
so copy the artifact to Drive and commit `docs/tribe_environment.md` from a
machine with push access.


In [ ]:
import shutil
from pathlib import Path

REPO_DIR = Path("/content/neurotutorsim")
artifact = REPO_DIR / "outputs" / "verification" / "gate17.json"
if artifact.exists():
    shutil.copy2(artifact, PROJECT_DRIVE / "gate17.json")
    print(f"copied {artifact} -> {PROJECT_DRIVE / 'gate17.json'}")
else:
    print("no gate17.json -- verification did not pass")

print((REPO_DIR / "docs" / "tribe_environment.md").read_text()[:800])
